In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# from google.colab import files
# uploaded = files.upload()

In [ ]:
!pip install earthengine-api pystac


In [ ]:
import rasterio
import os
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import mapping, box, Polygon
import requests
import xml.etree.ElementTree as ET
from io import BytesIO

import sys
sys.path.append('..')
#import constants
import csv

import pystac
from pystac.extensions.table import TableExtension
from urllib.parse import urlparse, parse_qs
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension
from pystac.catalog import CatalogType
import datetime
import ee
ee.Authenticate()
ee.Initialize(project='ee-corestackdev')

In [ ]:
#STAC_SAVE_DIR = "/home/vishnu/STAC-spec-raster/STAC-spec/data/PANindia_stac_raster_test"
STAC_SAVE_DIR = "/content/drive/MyDrive/STAC_spec_PANIndia_test"

SUB_COLLECTIONS = {}

MAIN_COLLECTION_ID = "panindia-layers"

In [ ]:


def load_sheet_df(sheet_id):

    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&gid=0"
    df = pd.read_csv(url)
    df = df.dropna(how='all')

    cleaned_cols = []
    for col in df.columns:
        cleaned_col = re.sub(r'\s+', ' ', col.strip())
        cleaned_cols.append(cleaned_col)

    df.columns = cleaned_cols

    print(df.columns.tolist())

    REQUIRED_COLUMNS = ['collection_name', 'display_name', 'Layer Name', 'ee_layer_name', 'GEE asset link', 'Style file url']

    missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Missing required columns in the sheet: {', '.join(missing_cols)}. "
            f"Please ensure the exact header names in your Google Sheet match the following list: {REQUIRED_COLUMNS}"
        )

    STYLE_URL_COLUMN = 'Style file url'
    LAYER_NAME_COLUMN = 'Layer Name'

    df = df.dropna(subset=[LAYER_NAME_COLUMN]).reset_index(drop=True)

    return df, STYLE_URL_COLUMN

In [ ]:

sheet_id = '1ulwuZ0mEUpRYgchVlp9BLwTBs4eCp5PczR0zL79q1rA'
#sheet_id = "1rSg8Zm0RHQ7wgyVr7ZZoDj66CB88Om7gbXfqLhDK8HI"
#sheet_id='1sLskFtH-sbsWMEX7-oGlkwQCHu_OW7QaUdQSeRTGDIk'
df, STYLE_URL_COLUMN = load_sheet_df(sheet_id)


In [ ]:
def read_raster_data(image_info, ee_image):
    band_info = image_info['bands'][0]

    crs = band_info.get('crs')
    transform = band_info.get('crs_transform', [30, 0, 0, 0, -30, 0])
    bands = len(image_info['bands'])

    if "dimensions" in band_info:
        width = band_info['dimensions'][0]
        height = band_info['dimensions'][1]
    else:
        print(f"'dimensions' missing for asset. Estimating from geometry & scale")
        try:
            scale_x = abs(transform[0])
            scale_y = abs(transform[4])
            region = ee.Image(ee_image).geometry().bounds().getInfo()
            coords = region["coordinates"][0]
            min_x, min_y = coords[0]
            max_x, max_y = coords[2]
            width = int((max_x - min_x) / scale_x)
            height = int((max_y - min_y) / scale_y)
        except Exception as e:
            raise RuntimeError(f"Could not infer dimensions: {e}")

    min_x = transform[2]
    max_y = transform[5]
    max_x = min_x + width * transform[0]
    min_y = max_y + height * transform[4]

    bbox = [min_x, min_y, max_x, max_y]
    footprint = Polygon([
        [min_x, min_y], [min_x, max_y],
        [max_x, max_y], [max_x, min_y],
        [min_x, min_y]
    ])

    try:
        gsd_meters = ee_image.projection().nominalScale().getInfo()
    except Exception as e:
        print(f"Could not get nominal scale in meters for CRS {crs}. Fallback to crs_transform. Error: {e}")
        gsd_meters = abs(transform[0])

    shape = (bands, height, width)
    data_type = band_info['data_type']['type']

    return (bbox, mapping(footprint), crs, gsd_meters, shape, data_type)


In [ ]:


def normalize_key(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "-", text)  # replace any non-alphanumeric with hyphen
    text = re.sub(r"-+", "-", text)          # collapse multiple hyphens
    return text.strip("-")

In [ ]:
def load_layer_descriptions(csv_path="/content/drive/MyDrive/layer_descriptions.csv"):
    desc_map = {}

    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            norm_key = normalize_key(row["layer_name"])
            desc_map[norm_key] = row["layer_description"]

    return desc_map

LAYER_DESCRIPTIONS = load_layer_descriptions("/content/drive/MyDrive/layer_descriptions.csv")


In [ ]:
def create_raster_item(id, bbox, footprint, crs, shape):

    description_key = normalize_key(id)
    item_description = LAYER_DESCRIPTIONS.get(description_key, id)

    raster_item = pystac.Item(
        id=id,
        geometry=footprint,
        bbox=bbox,
        datetime=datetime.datetime.now(datetime.timezone.utc),
        properties={"description": item_description},
    )

    proj_ext = ProjectionExtension.ext(raster_item, add_if_missing=True)
    proj_ext.epsg = 4326
    proj_ext.shape = [shape[1], shape[2]]

    return (raster_item, None)


In [ ]:
def add_raster_data_asset(raster_item, url_string):
    raster_item.add_asset("data", Asset(
        href=url_string,
        roles=["data"],
        title="Asset Link"
    ))
    return raster_item

In [ ]:
def add_raster_extension(raster_item, gsd_val, data_type_val):
    raster_ext = RasterExtension.ext(raster_item.assets["data"], add_if_missing=True)
    raster_band = RasterBand.create(
        data_type=data_type_val,
        spatial_resolution=gsd_val,
    )
    raster_ext.bands = [raster_band]

In [ ]:
def parse_raster_style_file(style_file_url):

    try:
        response = requests.get(style_file_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise


    try:
        xml_string = response.content.decode('utf-8', errors='replace')
        style_content = BytesIO(xml_string.encode('utf-8'))
        tree = ET.parse(style_content)
    except ET.ParseError as e:
        try:
            print(f"Warning: Failed to parse XML with UTF-8 ({e}). Trying ISO-8859-1.")
            xml_string_alt = response.content.decode('iso-8859-1', errors='ignore')
            style_content = BytesIO(xml_string_alt.encode('utf-8'))
            tree = ET.parse(style_content)
        except ET.ParseError as e_final:
            raise e_final

    root = tree.getroot()
    classes = []

    for entry in root.findall(".//paletteEntry") + root.findall(".//item"):
        class_info = {}
        for attr_key, attr_value in entry.attrib.items():
            if attr_key == "value":
                try:
                    class_info[attr_key] = int(attr_value)
                except ValueError:
                    class_info[attr_key] = attr_value
            else:
                class_info[attr_key] = attr_value
        if class_info:
            classes.append(class_info)

    if not classes:
         raise ValueError(f"No classification data (paletteEntry or item tags) found in style file.")

    return classes

In [ ]:
def add_classification_extension(raster_style_url, raster_item):

    style_info = parse_raster_style_file(style_file_url=raster_style_url)
    classification_ext = ClassificationExtension.ext(raster_item.assets["data"], add_if_missing=True)
    stac_classes = []

    for cls in style_info:
        color_hint = cls['color'].replace('#','') if cls.get('color') else None

        value = cls.get("value")
        if not isinstance(value, int):
             try:
                 value = int(float(str(value))) if value is not None else None
             except (TypeError, ValueError):
                 value = None

        if value is None:
             print(f"Classification skipped for an entry with invalid value.")
             continue

        stac_class_obj = Classification.create(
            value=value,
            name=cls.get("label") or f"Class {value}",
            description=cls.get("label"),
            color_hint=color_hint
        )
        stac_classes.append(stac_class_obj)

    if stac_classes:
        classification_ext.classes = stac_classes
    else:
        raster_item.assets["data"].properties.pop("classification:classes", None)


    return (raster_item,style_info)

In [ ]:
def add_stylefile_asset(raster_item, style_file_url):
    raster_item.add_asset("style", Asset(
        href=style_file_url,
        media_type=MediaType.XML,
        roles=["metadata", "visualisation"],
        title="QGIS Style file"
    ))
    return raster_item

In [ ]:
def generate_raster_stac(image_info, url_string, style_file_url, layer_name, gsd, data_type, bbox, footprint, crs, shape):

    stac_id = layer_name.replace(" ", "_").replace("/", "_").replace("-", "_").lower()


    raster_item, _ = create_raster_item(id=stac_id, bbox=bbox, footprint=footprint, crs=crs, shape=shape)

    raster_item.properties["title"] = layer_name

    raster_item = add_raster_data_asset(raster_item, url_string=url_string)

    add_raster_extension(raster_item, gsd_val=gsd, data_type_val=data_type)


    raster_item, style_info = add_classification_extension(raster_style_url=style_file_url,
                                                             raster_item=raster_item)

    raster_item = add_stylefile_asset(raster_item, style_file_url=style_file_url)

    return raster_item, style_info

In [ ]:

OUTPUT_DIR = '/content/drive/MyDrive/STAC_spec_PANIndia_test'
STAC_CATALOG_FILENAME = 'catalog.json'

def create_root_and_collection():

    root_catalog_href = os.path.join(OUTPUT_DIR, STAC_CATALOG_FILENAME)


    root_catalog = pystac.Catalog(
        id='PanIndiaCatalogs',
        description='This spatio temporal asset catalog contains all data layers of CoRE Stack (https://core-stack.org/) generated at Pan India level."',
        title='Pan India Spatio Temporal Asset Catalog'
    )


    root_catalog.set_self_href(root_catalog_href)


    panindia_collection_id = 'panindia-layers'


    extent = pystac.Extent(
        spatial=pystac.SpatialExtent(bboxes=[[-180.0, -90.0, 180.0, 90.0]]),
        temporal=pystac.TemporalExtent(intervals=[[datetime.datetime(2000, 1, 1), None]])
    )

    panindia_collection = pystac.Collection(
        id=panindia_collection_id,
        description='This spatio temporal asset catalog contains all data layers of CoRE Stack (https://core-stack.org/) generated at Pan India level.',
        extent=extent,
        title='Pan India Spatio Temporal Asset Catalog'
    )

    root_catalog.add_child(panindia_collection)

    print("STAC Root Catalog and Base Collection created.")
    return root_catalog, panindia_collection

In [ ]:
def create_sub_collection(title, parent_collection):

    title_str = str(title) if title is not None else "unknown"
    collection_id = title.lower().replace(' ', '-').replace(':', '').replace('/', '-')

    description = f"STAC collection for {title_str} of Pan India."

    if collection_id not in SUB_COLLECTIONS:
        print(f"Creating new Sub-Collection (Theme): {title} ---")

        sub_collection = pystac.Collection(
            id=collection_id,
            title=title,
            description=description,
            extent=parent_collection.extent,
            license=parent_collection.license,
            providers=parent_collection.providers
        )


        parent_collection.add_child(sub_collection)
        SUB_COLLECTIONS[collection_id] = sub_collection

    return SUB_COLLECTIONS[collection_id]

In [ ]:
def parse_asset_id(url_string, row_index):
    try:
        parsed_url = urlparse(url_string)
        query_params = parse_qs(parsed_url.query)
        return query_params['asset'][0]
    except Exception as e:
        print(f"Skipping row {row_index}: Could not parse asset ID from URL: {url_string}. Error: {e}")
        return None

In [ ]:
def load_gee_image(asset_id, mosaic_if_collection=True):

    try:
        asset_metadata = ee.data.getAsset(asset_id)
        asset_type = asset_metadata.get('type')

        if asset_type == 'IMAGE':
            print("Confirmed asset type: ee.Image.")
            return ee.Image(asset_id)

        elif asset_type == 'IMAGE_COLLECTION':
            print("Confirmed asset type: ee.ImageCollection.")
            collection = ee.ImageCollection(asset_id)

            if mosaic_if_collection:

                first_img = ee.Image(collection.first())
                proj = first_img.projection()
                scale = proj.nominalScale()
                crs = proj.crs()

                print(f"Using projection from first image: {crs}, scale: {scale.getInfo()} m")


                mosaicked = collection.mosaic().reproject(crs=crs, scale=scale)
                return mosaicked
            else:
                return collection

        elif asset_type == 'TABLE':
            print("Confirmed asset type: ee.FeatureCollection.")
            return ee.FeatureCollection(asset_id)

        else:
            print(f"Unsupported asset type: {asset_type}. Skipping.")
            return None

    except Exception as e:
        print(f"Failed to load asset {asset_id}: {e}")
        return None


In [ ]:

def process_layer(index, row, style_col_name, target_collection):

    original_layer_name = str(row['Layer Name']).strip()
    ee_layer_name = str(row['ee_layer_name']).strip()
    asset_link = str(row['GEE asset link']).strip()
    display_name = str(row['display_name']).strip() if not pd.isna(row['display_name']) else original_layer_name

    unique_name_part = ee_layer_name.lower()


    unique_name_part = re.sub(r'[\s\.\-]+', '_', unique_name_part).strip('_')


    item_id = f"{original_layer_name}_{unique_name_part}".lower()


    style_url = str(row[style_col_name]).strip() if not pd.isna(row[style_col_name]) else None

    try:
        bbox = [-180.0, -90.0, 180.0, 90.0]
    except Exception as e:
        print(f"Warning: Could not extract bounds for {original_layer_name}. Using default. Error: {e}")
        bbox = [-180.0, -90.0, 180.0, 90.0]



    item = pystac.Item(
        id=item_id,
        geometry=None,
        bbox=bbox,
        datetime=datetime.datetime.now(),
        properties={
            'title': display_name,
            'ee_layer_name': ee_layer_name,
            'style_file_url': style_url if style_url else 'No custom style'
        }
    )

    if style_url:
        item.add_asset(
            key='style_file',
            asset=pystac.Asset(
                href=style_url,
                media_type=pystac.MediaType.JSON,
                roles=['metadata', 'style']
            )
        )

    # Create a pystac Asset for the Earth Engine layer
    asset_type_raw = 'Image'
    if 'ImageCollection' in ee_layer_name:
        asset_type_raw = 'ImageCollection'
    elif 'FeatureCollection' in ee_layer_name:
        asset_type_raw = 'FeatureCollection'

    ee_asset_metadata = {
        'gee:type': asset_type_raw,
        'gee:assets': [{'name': ee_layer_name, 'type': f'ee:{asset_type_raw}'}]
    }

    item.add_asset(
        key='gee_asset',
        asset=pystac.Asset(
            href=asset_link,
            media_type=pystac.MediaType.TEXT,
            roles=['data'],
            title='Earth Engine Asset Link',
            extra_fields=ee_asset_metadata
        )
    )


    target_collection.add_item(item)
    return 1

In [ ]:

def save_catalog(catalog, item_count):
    OUTPUT_DIR = '/content/drive/MyDrive/STAC_spec_PANIndia_test'

    print(f"Attempting to save STAC Catalog to: {OUTPUT_DIR}")


    if not os.path.exists(OUTPUT_DIR):
        try:
            os.makedirs(OUTPUT_DIR)
            print(f"Created directory: {OUTPUT_DIR}")
        except Exception as e:
            print(f"ERROR: Could not create output directory {OUTPUT_DIR}. Check Google Drive permissions. Error: {e}")
            return


    try:
        catalog.save(
            catalog_type=pystac.CatalogType.SELF_CONTAINED,
            dest_href=OUTPUT_DIR,

        )
        print("SUCCESS: STAC Catalog files written successfully.")
        print(f"Total Items Generated: {item_count}")
        print(f"Files should be located in: {OUTPUT_DIR}")

    except Exception as e:
        print(f"CRITICAL ERROR: Failed to save STAC Catalog using PySTAC. Error: {e}")
        print("Please check if your Google Drive is mounted correctly and the path is valid.")


In [ ]:
def run_stac_generation(df, panindia_collection, root_catalog, STYLE_URL_COLUMN):
    generated_items_count = 0


    collection_groups = df.groupby('collection_name')


    created_collections = {panindia_collection.id: panindia_collection}


    for collection_name, group_df in collection_groups:


        collection_id = collection_name.strip().lower().replace(' ', '-').replace('_', '-')
        collection_title = collection_name.strip()


        if collection_id == panindia_collection.id:
            target_collection = panindia_collection
        else:
            if collection_id not in created_collections:
                print(f"\n--- NEW COLLECTION: Creating STAC Collection for '{collection_title}' ({collection_id}) ---")

                extent = pystac.Extent(
                    spatial=pystac.SpatialExtent(bboxes=[[-180.0, -90.0, 180.0, 90.0]]),
                    temporal=pystac.TemporalExtent(intervals=[[datetime.datetime(2000, 1, 1), None]])
                )

                sub_collection = pystac.Collection(
                    id=collection_id,
                    description=f"Layers related to {collection_title}.",
                    extent=extent,
                    title=collection_title
                )
                root_catalog.add_child(sub_collection)
                created_collections[collection_id] = sub_collection

            target_collection = created_collections[collection_id]


        print(f"\nProcessing {len(group_df)} layers for collection: {collection_title}...")


        for index, row in group_df.iterrows():
            layer_name = row['Layer Name'].strip()
            ee_layer_name = row['ee_layer_name'].strip()
            display_name = row['display_name'].strip()


            print(f"  -> Processing Item: '{layer_name}' (Display Name: '{display_name}')")
            print(f"     EE Layer Name: {ee_layer_name}")


            item_count = process_layer(
                index,
                row,
                STYLE_URL_COLUMN,
                target_collection
            )
            generated_items_count += item_count


    print(f"Completed. Total {generated_items_count} STAC Items generated.")
    save_catalog(root_catalog, generated_items_count)

In [ ]:
df, STYLE_URL_COLUMN = load_sheet_df(sheet_id)
root_catalog, panindia_collection = create_root_and_collection()
run_stac_generation(df, panindia_collection, root_catalog, STYLE_URL_COLUMN)